# Pasture Studies — reproducible training and evaluation

This notebook runs the approved **synthetic** cattle-policy experiments. It keeps the NumPy baseline and failed MVP1 report intact. It does not establish biological validity, and it does not claim GPU acceleration unless the recorded benchmark supports that claim.

**Run in Kaggle:** create a new notebook from this file; select the Pasture Studies GitHub source (or use the clone cell), Python, Internet enabled for clone/package install, and accelerator `T4 x2` if available. Kaggle GPU quota/access is account-dependent. The notebook checks the actual visible devices and falls back only when explicitly configured. Save a Kaggle notebook version with “Save & Run All” to preserve `/kaggle/working` outputs. Do not publish credentials or push artifacts back to GitHub.

The matched algorithm comparison is CPU REINFORCE vs CPU A2C with the same seed/config. GPU runs are A2C-only for device comparison, as specified by the plan. The smoke run is a pipeline check, not a scientific evaluation.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, time, platform, re

REPO_URL = "https://github.com/santleme/pasture-studies.git"
# Immutable pipeline source commit published as the base of this notebook pin.
PASTURE_PIPELINE_COMMIT = "74f5ccd5b6dd785b2e424d29302c15bcec73bbc1"
if not re.fullmatch(r"[0-9a-f]{40}", PASTURE_PIPELINE_COMMIT):
    raise RuntimeError("The published pipeline commit pin is invalid")
PASTURE_REF = PASTURE_PIPELINE_COMMIT
EXPECTED_COMMIT = PASTURE_PIPELINE_COMMIT
PASTURE_NOTEBOOK_REVISION = os.environ.get("PASTURE_NOTEBOOK_REVISION", "unspecified")
WORK_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".").resolve()
REPO_DIR = WORK_ROOT / "pasture-studies"
RUN_ID = os.environ.get("PASTURE_RUN_ID", "mvp1-run")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", RUN_ID):
    raise ValueError("PASTURE_RUN_ID must be a safe 1-64 character identifier")
ARTIFACT_ROOT = WORK_ROOT / f"pasture-training-artifacts-{RUN_ID}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None, check=True, env=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check, text=True, env=env)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Work root:", WORK_ROOT)
print("Run ID:", RUN_ID)
print("Artifacts:", ARTIFACT_ROOT)


## 1. Fetch the exact reviewed pipeline commit and record provenance

This notebook is pinned to one immutable 40-character pipeline SHA. The publication notebook revision is separately recorded for provenance. Do not override the pipeline pin for confirmatory runs. `PASTURE_RUN_ID` defaults to `mvp1-run`; keep the same ID and artifact directory to resume, or choose a new ID for a separate run. Resuming after a Kaggle session reset additionally requires that the prior `/kaggle/working` artifacts were retained or restored from a downloaded archive.


In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--no-checkout", "--no-tags", REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXPECTED_COMMIT])
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Resolved source commit:", commit)
if commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Expected {EXPECTED_COMMIT}, got {commit}; refusing to continue")
source_record = ARTIFACT_ROOT / "source_commit.txt"
if source_record.exists():
    old = dict(line.split("=", 1) for line in source_record.read_text().splitlines() if "=" in line)
    if old.get("commit") != commit:
        raise RuntimeError("Existing run artifacts are bound to a different source commit; use a new PASTURE_RUN_ID")
else:
    source_record.write_text(f"repo={REPO_URL}\nref={PASTURE_REF}\ncommit={commit}\n")
os.chdir(REPO_DIR)


## 2. Install the optional training environment

The base simulator remains NumPy-first. This notebook installs the repo's optional, pinned training requirements. Internet access is needed unless those wheels are already available in the Kaggle image.

In [ ]:
req = REPO_DIR / "requirements-training.txt"
if not req.exists():
    raise FileNotFoundError(f"Missing optional training requirements: {req}")
run([sys.executable, "-m", "pip", "install", "-r", req])
run([sys.executable, "-m", "pip", "check"])
run([sys.executable, "-m", "validation.train_torch_mvp1", "--help"])

## 3. Freeze the outcome-blind seed protocol and detect accelerator

The generator binds the approved plan, historical failure report, trainer/evaluator, tests, requirements, this notebook, and this guide. It reads no model outcomes. Run it only from the immutable reviewed source commit and before any training/evaluation outcomes; its hash is copied into every report/export. The generated manifest is the seed authority for this run. The checked-in preregistration is provenance evidence, but a runtime-generated copy is used because the source-bound hashes must match this exact checkout.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

# Generate the preregistration before collecting outcomes. This fails closed if
# the approved artifacts/source hashes do not match the checked-out revision.
PREREG_PATH = ARTIFACT_ROOT / "preregistration.json"
run([sys.executable, "validation/preregister_torch_mvp1.py", "--output", PREREG_PATH], cwd=REPO_DIR)
PREREG = json.loads(PREREG_PATH.read_text())
if PREREG.get("status") != "PRE_OUTCOME_PREREGISTRATION_PENDING_PILOT_DELTA0":
    raise RuntimeError(f"Unexpected preregistration status: {PREREG.get('status')}")
PREREG_SHA256 = PREREG["manifest_sha256"]
TRAIN_SEEDS = list(PREREG["seed_splits"]["training_root_seeds"])
EPISODES_PER_SEED = int(PREREG["training_config"]["episodes_per_train_seed"])
if len(TRAIN_SEEDS) != 10 or len(set(TRAIN_SEEDS)) != len(TRAIN_SEEDS):
    raise RuntimeError("Unexpected training seed roots in preregistration")
if EPISODES_PER_SEED != 278:
    raise RuntimeError(f"Unexpected preregistered episode count: {EPISODES_PER_SEED}")
FIXED_CORPUS_SEED = 51337  # separate compute fixture seed, not a policy-training seed
CHECKPOINT_EVERY = 16
RUN_FULL = True
RUN_GPU_BENCHMARK = True
seed_record = {"manifest_path": str(PREREG_PATH), "manifest_sha256": PREREG_SHA256,
               "status": PREREG["status"], "training_root_seeds": TRAIN_SEEDS,
               "episodes_per_train_seed": EPISODES_PER_SEED,
               "validation_split": "from seed manifest; 30 worlds per scenario family",
               "fixed_corpus_seed": FIXED_CORPUS_SEED}
(ARTIFACT_ROOT / "preregistration_binding.json").write_text(json.dumps(seed_record, indent=2) + "\n")
print("Preregistration SHA-256:", PREREG_SHA256)
print("Training root seeds:", TRAIN_SEEDS)
print("Episodes per training root:", EPISODES_PER_SEED)
if not torch.cuda.is_available():
    print("No CUDA: CPU comparison can run; GPU measurement remains NOT RUN.")
elif torch.cuda.device_count() < 2:
    print("Fewer than two visible GPUs: no T4x2 result can be claimed.")


## 4. Smoke test before long training

This disposable smoke run verifies the trainer/checkpoint path only; it is not a preregistered training arm and its checkpoint is never evaluated or promoted. Formal training loops below use only root seeds and episode counts read from the frozen manifest. Inspect the smoke outputs before the expensive runs.


In [ ]:
SMOKE_DIR = ARTIFACT_ROOT / "smoke"
SMOKE_METRICS = SMOKE_DIR / "metrics.jsonl"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
if any(SMOKE_DIR.rglob("LATEST")):
    if not SMOKE_METRICS.exists():
        raise RuntimeError("Existing smoke checkpoint lacks its metrics file; recover the artifact directory before continuing")
    print("Existing smoke checkpoint and metrics found; skipping duplicate smoke updates.")
else:
    run([sys.executable, "-m", "validation.train_torch_mvp1", "--smoke",
         "--algorithm", "both", "--device", "cpu", "--seed", str(TRAIN_SEEDS[0]),
         "--output", SMOKE_DIR, "--metrics", SMOKE_METRICS,
         "--checkpoint-every", "1"])
print("Smoke outputs:", sorted(p.name for p in SMOKE_DIR.iterdir()))
if not SMOKE_METRICS.exists():
    raise RuntimeError("Smoke run did not emit the requested metrics file")


## 5. Matched CPU comparison: trajectory REINFORCE vs vanilla A2C

Both algorithms run on CPU with the same seed list, episodes, environment semantics, observations and run cadence. Keep each seed's output isolated. Do not select the better seed or alter rewards based on these results. The two algorithms are separate experimental conditions; this notebook reports their measured outcomes without asserting A2C is superior.

In [ ]:
CPU_ROOT = ARTIFACT_ROOT / "matched_cpu"
CPU_ROOT.mkdir(parents=True, exist_ok=True)

def record_run_timing(outdir, timing):
    outdir = Path(outdir)
    timing_path = outdir / "run_timing.json"
    segment_path = outdir / "run_timing_segments.jsonl"
    if timing["resumed"]:
        # A continuation's wall time is not the original full-run duration.
        # Preserve a prior complete record and keep this attempt separate.
        timing["complete_full_run_timing"] = False
        timing["elapsed_seconds_current_invocation"] = timing.pop("elapsed_seconds")
        timing["elapsed_seconds"] = None
        with segment_path.open("a") as stream:
            stream.write(json.dumps(timing, sort_keys=True) + "\n")
        if not timing_path.exists():
            timing_path.write_text(json.dumps(timing, indent=2) + "\n")
        return
    if timing_path.exists():
        # Never replace the original full-run duration with a later no-op resume.
        with segment_path.open("a") as stream:
            stream.write(json.dumps({**timing, "complete_full_run_timing": False,
                                     "elapsed_seconds_current_invocation": timing["elapsed_seconds"]},
                                    sort_keys=True) + "\n")
        return
    timing["complete_full_run_timing"] = True
    timing_path.write_text(json.dumps(timing, indent=2) + "\n")

def train_one(algorithm, device, seed, outdir, episodes=EPISODES_PER_SEED, resume=None):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    metrics = outdir / "metrics.jsonl"
    checkpoint_root = outdir / algorithm
    resume_target = resume if resume is not None else ("latest" if (checkpoint_root / "LATEST").is_file() else None)
    cmd = [sys.executable, "-m", "validation.train_torch_mvp1",
           "--mode", "train", "--algorithm", algorithm, "--device", device,
           "--seed", str(seed), "--episodes", str(episodes),
           "--output", str(outdir), "--metrics", str(metrics),
           "--checkpoint-every", str(CHECKPOINT_EVERY)]
    if resume_target is not None:
        cmd += ["--resume", str(resume_target)]
    started = time.perf_counter()
    run(cmd)
    elapsed = time.perf_counter() - started
    if not metrics.exists():
        raise RuntimeError(f"Missing metrics after {algorithm}/{seed}: {metrics}")
    timing = {"phase": "dynamic_end_to_end", "algorithm": algorithm,
              "device": device, "world_size": 1, "seed": int(seed),
              "episodes": int(episodes), "elapsed_seconds": elapsed,
              "includes_process_startup_checkpoint_and_final_eval": True,
              "resumed": resume_target is not None, "source_commit": commit}
    record_run_timing(outdir, timing)
    return metrics

if RUN_FULL:
    for seed in TRAIN_SEEDS:
        for alg in ("reinforce", "a2c"):
            train_one(alg, "cpu", seed, CPU_ROOT / alg / f"seed-{seed}")
else:
    print("RUN_FULL=False; matched comparison skipped")


## 6. A2C accelerator workload (single T4 and T4x2 when available)

This is the approved device comparison arm; it is A2C only. The trainer supports `torchrun` DDP with NCCL, rank-sharded complete episodes and rank-zero checkpointing. On T4x2, launch both ranks with `torchrun --standalone --nproc_per_node=2`; the trainer must use both visible devices. A single-T4 run sets `CUDA_VISIBLE_DEVICES=0` and is a separate condition. Fixed-work and dynamic end-to-end results remain distinct; a successful run or GPU visibility alone is not an acceleration result.


In [ ]:
GPU_ROOT = ARTIFACT_ROOT / "a2c_accelerator"
if RUN_FULL and RUN_GPU_BENCHMARK and torch.cuda.is_available():
    if torch.cuda.device_count() >= 1:
        # T4x1 condition: hide the second GPU even when Kaggle exposes T4x2.
        for seed in TRAIN_SEEDS:
            out = GPU_ROOT / "t4x1" / f"seed-{seed}"
            env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
            metrics = out / "metrics.jsonl"; out.mkdir(parents=True, exist_ok=True)
            resuming = (out / "a2c" / "LATEST").is_file()
            started = time.perf_counter()
            run([sys.executable, "-m", "validation.train_torch_mvp1",
                 "--mode", "train", "--algorithm", "a2c", "--device", "cuda",
                 "--seed", seed, "--episodes", EPISODES_PER_SEED,
                 "--output", out, "--metrics", metrics,
                 "--checkpoint-every", CHECKPOINT_EVERY] +
                (["--resume", "latest"] if resuming else []),
                cwd=REPO_DIR, env=env)
            record_run_timing(out, {"phase": "dynamic_end_to_end", "algorithm": "a2c",
                "device": "cuda", "world_size": 1, "seed": int(seed), "episodes": int(EPISODES_PER_SEED),
                "elapsed_seconds": time.perf_counter() - started,
                "includes_process_startup_checkpoint_and_final_eval": True,
                "resumed": resuming, "source_commit": commit})
    if torch.cuda.device_count() >= 2:
        for seed in TRAIN_SEEDS:
            out = GPU_ROOT / "t4x2" / f"seed-{seed}"
            out.mkdir(parents=True, exist_ok=True); metrics = out / "metrics.jsonl"
            resuming = (out / "a2c" / "LATEST").is_file()
            started = time.perf_counter()
            run(["torchrun", "--standalone", "--nproc_per_node=2", "-m",
                 "validation.train_torch_mvp1", "--mode", "train",
                 "--algorithm", "a2c", "--device", "cuda", "--seed", seed,
                 "--episodes", EPISODES_PER_SEED, "--output", out,
                 "--metrics", metrics, "--checkpoint-every", CHECKPOINT_EVERY] +
                (["--resume", "latest"] if resuming else []))
            record_run_timing(out, {"phase": "dynamic_end_to_end", "algorithm": "a2c",
                "device": "cuda", "world_size": 2, "seed": int(seed), "episodes": int(EPISODES_PER_SEED),
                "elapsed_seconds": time.perf_counter() - started,
                "includes_process_startup_checkpoint_and_final_eval": True,
                "resumed": resuming, "source_commit": commit})
    else:
        print("T4x2 distributed run NOT RUN: fewer than two visible CUDA devices.")
else:
    print("A2C CUDA benchmark NOT RUN (disabled or no CUDA).")


## 7. Dynamic end-to-end training/pipeline timings (not fixed-work attribution)

The prior training cells record one elapsed wall time for each pristine full run, including process startup, environment rollouts, checkpointing and final evaluation. Resume invocations are written separately and excluded from paired dynamic timing so a continuation cannot masquerade as a complete run. Compare paired seeds and report it separately from the fixed-corpus compute experiment below. The dynamic pipeline acceptance gate also requires trajectory/action disagreement and simulator-work counters (resource retries, navigation target candidates, BFS expansions, path cells) within the approved limits. The current training CLI timing record alone does not contain those paired divergence counters; therefore this notebook will report durations but keep the end-to-end pipeline verdict **INCONCLUSIVE**, even if a GPU run is faster. Same seed does not imply identical trajectories across devices.

In [ ]:
import numpy as np
dynamic_timing = {}
for condition in ("cpu", "t4x1", "t4x2"):
    rows = []
    for seed in TRAIN_SEEDS:
        if condition == "cpu":
            candidates = [CPU_ROOT / alg / f"seed-{seed}" / "run_timing.json" for alg in ("a2c",)]
        else:
            candidates = [GPU_ROOT / condition / f"seed-{seed}" / "run_timing.json"]
        path = next((p for p in candidates if p.exists()), None)
        if path:
            row = json.loads(path.read_text())
            if (row.get("complete_full_run_timing") is True and row.get("resumed") is False
                    and isinstance(row.get("elapsed_seconds"), (int, float))):
                row["condition"] = condition
                rows.append(row)
            else:
                print("Excluding resume/segment-only timing from dynamic comparisons:", path)
    dynamic_timing[condition] = rows
    print(condition, "eligible pristine full-run timings", len(rows), "median seconds",
          float(np.median([r["elapsed_seconds"] for r in rows])) if rows else None)

cpu_by_seed = {r["seed"]: r["elapsed_seconds"] for r in dynamic_timing["cpu"]}
for condition in ("t4x1", "t4x2"):
    paired = [(cpu_by_seed[r["seed"]] / r["elapsed_seconds"], r["seed"])
              for r in dynamic_timing[condition] if r["seed"] in cpu_by_seed and r["elapsed_seconds"] > 0]
    print(condition, "paired CPU/GPU elapsed ratios (descriptive):", paired)
    dynamic_timing[f"cpu_vs_{condition}_paired_ratios"] = paired
    if len(paired) == len(TRAIN_SEEDS):
        print(condition, "dynamic end-to-end pipeline benefit verdict: INCONCLUSIVE — required paired action/trajectory and simulator-work divergence counters are not emitted by this timing record")
    else:
        print(condition, "dynamic end-to-end pipeline benefit verdict: UNMEASURED/INCOMPLETE — missing paired full runs")


## 8. Fixed-corpus GPU compute benchmark (separate from training)

This is the approved `kaggle-a2c-fixed-corpus-v1` workload: one frozen training-only corpus shared byte-for-byte by CPU, T4x1 and T4x2; 10 partitions × 278 complete episodes; 180 real float32 forward/loss/backward/clip/optimizer updates per device repetition; five paired repetitions. Corpus generation happens outside benchmark timing. The corpus is stale/off-policy during replay, so its weights and outputs are **never a learning result or promotable policy**. The benchmark CLI reports descriptive timings and does not itself certify acceptance. A visible GPU, successful smoke, or network microbenchmark alone is never a PASS.

In [ ]:
from pathlib import Path
FIXED_ROOT = ARTIFACT_ROOT / "fixed_corpus_benchmark"
FIXED_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS = FIXED_ROOT / "kaggle-a2c-fixed-corpus-v1.npz"
CORPUS_SEED = 51337
FIXED_REPEATS = 5
RUN_FIXED_BENCHMARK = RUN_FULL

# Confirm the merged engine exposes the approved subcommands before creating the full corpus.
run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "--help"], cwd=REPO_DIR)
if RUN_FIXED_BENCHMARK:
    run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "generate-corpus",
         "--seed", CORPUS_SEED, "--output", CORPUS], cwd=REPO_DIR)
    run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
         "--corpus", CORPUS, "--device", "cpu", "--repeats", FIXED_REPEATS,
         "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "cpu.json"], cwd=REPO_DIR)
    if torch.cuda.is_available():
        env1 = os.environ.copy(); env1["CUDA_VISIBLE_DEVICES"] = "0"
        run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
             "--corpus", CORPUS, "--device", "cuda", "--repeats", FIXED_REPEATS,
             "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "t4x1.json"], cwd=REPO_DIR, env=env1)
        if torch.cuda.device_count() >= 2:
            run(["torchrun", "--standalone", "--nproc_per_node=2",
                 "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
                 "--corpus", CORPUS, "--device", "cuda", "--repeats", FIXED_REPEATS,
                 "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "t4x2.json"], cwd=REPO_DIR)
        else:
            print("T4x2 fixed-corpus run NOT RUN: fewer than two CUDA devices.")
    else:
        print("CUDA fixed-corpus runs NOT RUN; CPU timing is not a GPU benchmark.")
else:
    print("Full fixed-corpus benchmark disabled; smoke results cannot satisfy acceptance.")


### Fixed-work results and verdict boundaries

The approved direct GPU compute gate requires five paired full repeats, identical corpus/source/config hashes, DDP/global-mean and resume/parity tests, and the predeclared speed criteria: GPU paired throughput speedup 95% CI lower bound ≥2.0 plus ≥30% elapsed reduction versus CPU; T4x2 additionally ≥1.20 versus T4x1. Threshold failure after a complete valid run is a measured **FAIL** for that device. Missing runs/hash parity/required validation is **INCONCLUSIVE/UNMEASURED**. Meeting timing alone remains only a candidate: this notebook deliberately leaves the final acceptance verdict **INCONCLUSIVE** unless the independent protocol, DDP parity/recovery and provenance gates are also verified. It cannot pass on availability or a microbenchmark.

In [ ]:
# Pair repeats and summarize fixed-work speed ratios. This reports evidence; it never
# converts device visibility or a partial run into an acceptance PASS.
import numpy as np
fixed_reports = {}
for name in ("cpu", "t4x1", "t4x2"):
    path = FIXED_ROOT / f"{name}.json"
    if path.exists(): fixed_reports[name] = json.loads(path.read_text())

def compare_fixed(first, second, out_name):
    a = FIXED_ROOT / f"{first}.json"; b = FIXED_ROOT / f"{second}.json"
    if a.exists() and b.exists():
        out = FIXED_ROOT / out_name
        run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "compare",
             "--first", a, "--second", b, "--output", out], cwd=REPO_DIR)
        return json.loads(out.read_text())
    return None

comparisons = {"cpu_vs_t4x1": compare_fixed("cpu", "t4x1", "cpu-vs-t4x1.json"),
               "cpu_vs_t4x2": compare_fixed("cpu", "t4x2", "cpu-vs-t4x2.json"),
               "t4x1_vs_t4x2": compare_fixed("t4x1", "t4x2", "t4x1-vs-t4x2.json")}
for name, report in fixed_reports.items():
    print(name, "workload=", report.get("workload"), "device=", report.get("device"),
          "world_size=", report.get("torchrun_world_size"), "runs=", len(report.get("runs", [])),
          "median seconds=", report.get("summary", {}).get("duration_seconds", {}).get("median"),
          "median decisions/s=", report.get("summary", {}).get("decisions_per_second", {}).get("median"),
          "corpus hash=", report.get("corpus_config", {}).get("corpus_sha256"))

def full_device_report(name, expected_device, expected_world_size):
    report = fixed_reports.get(name, {})
    runs = report.get("runs", [])
    return (len(runs) == FIXED_REPEATS
            and report.get("device") == expected_device
            and report.get("torchrun_world_size") == expected_world_size
            and bool(runs) and all(r.get("world_size") == expected_world_size for r in runs)
            and not runs[0].get("smoke_mode", False)
            and (expected_device != "cuda" or len(report.get("cuda_devices") or []) == expected_world_size))
def full_device_report(name, expected_device, expected_world_size):
    report = fixed_reports.get(name, {})
    runs = report.get("runs", [])
    return (len(runs) == FIXED_REPEATS
            and report.get("device") == expected_device
            and report.get("torchrun_world_size") == expected_world_size
            and bool(runs) and all(r.get("world_size") == expected_world_size for r in runs)
            and not runs[0].get("smoke_mode", False)
            and (expected_device != "cuda" or len(report.get("cuda_devices") or []) == expected_world_size)
            and (name != "t4x2" or report.get("ddp_active") is True))

hash_by_device = {name: report.get("corpus_config", {}).get("corpus_sha256")
                  for name, report in fixed_reports.items()}
def same_corpus(*names):
    values = [hash_by_device.get(name) for name in names]
    return all(values) and len(set(values)) == 1

compute_thresholds = {
    "t4x1_lower_ci_ge_2": (ci_lower(comparisons["cpu_vs_t4x1"]) or 0) >= 2.0,
    "t4x1_elapsed_reduction_ge_30pct": (paired_speed(comparisons["cpu_vs_t4x1"]) or 0) >= 1 / 0.70,
    "t4x2_lower_ci_ge_2": (ci_lower(comparisons["cpu_vs_t4x2"]) or 0) >= 2.0,
    "t4x2_elapsed_reduction_ge_30pct": (paired_speed(comparisons["cpu_vs_t4x2"]) or 0) >= 1 / 0.70,
    "t4x2_lower_ci_vs_t4x1_ge_1_20": (ci_lower(comparisons["t4x1_vs_t4x2"]) or 0) >= 1.20}
valid_t4x1_pair = (full_device_report("cpu", "cpu", 1) and full_device_report("t4x1", "cuda", 1)
                   and same_corpus("cpu", "t4x1"))
valid_t4x2_pair = (full_device_report("cpu", "cpu", 1) and full_device_report("t4x1", "cuda", 1)
                   and full_device_report("t4x2", "cuda", 2)
                   and same_corpus("cpu", "t4x1", "t4x2"))
print("Predeclared timing checks:", compute_thresholds)
if not RUN_FIXED_BENCHMARK or "cpu" not in fixed_reports:
    t4x1_compute_status = t4x2_compute_status = "UNMEASURED"
elif not valid_t4x1_pair:
    t4x1_compute_status = "INCONCLUSIVE (incomplete repeats/device/corpus match)"
elif not all(compute_thresholds[k] for k in ("t4x1_lower_ci_ge_2", "t4x1_elapsed_reduction_ge_30pct")):
    t4x1_compute_status = "FAIL (complete T4x1 fixed-work timing missed threshold)"
else:
    t4x1_compute_status = "INCONCLUSIVE (timing thresholds met; DDP parity/recovery/provenance review still required)"
if not RUN_FIXED_BENCHMARK or "cpu" not in fixed_reports:
    t4x2_compute_status = "UNMEASURED"
elif not valid_t4x2_pair:
    t4x2_compute_status = "INCONCLUSIVE (T4x2 full repeats, two-rank DDP, or shared corpus not verified)"
elif not all(compute_thresholds[k] for k in ("t4x2_lower_ci_ge_2", "t4x2_elapsed_reduction_ge_30pct", "t4x2_lower_ci_vs_t4x1_ge_1_20")):
    t4x2_compute_status = "FAIL (complete T4x2 fixed-work timing missed threshold)"
else:
    t4x2_compute_status = "INCONCLUSIVE (timing thresholds met; DDP parity/recovery/provenance review still required)"
print("GPU compute acceleration — T4x1 verdict:", t4x1_compute_status)
print("GPU compute acceleration — T4x2 verdict:", t4x2_compute_status)
print("This verdict is separate from dynamic end-to-end pipeline benefit and policy learning/cognition.")


## 9. Manifest-defined validation evaluation and paired cognition probes

The evaluator consumes the exact source-bound preregistration used to set training seeds. `--split validation` selects the manifest's 30 world seeds per scenario family; the same worlds are paired across trained, untrained, and heuristic arms and across available checkpoints. This is validation/diagnostic evidence only—not a locked-test result. Keep each seed/checkpoint report; do not select or tune using locked outcomes.

The current single-checkpoint evaluator reports paired memory/perception interventions, but it does not perform the complete preregistered multi-checkpoint clustered bootstrap/Holm analysis. A validation run therefore cannot pass the cognition gate or establish learning/biological validity.


In [ ]:
HELDOUT_ROOT = ARTIFACT_ROOT / "heldout_evaluation"
assert len(TRAIN_SEEDS) == 10
EVAL_CONDITIONS = [("cpu", "reinforce"), ("cpu", "a2c"), ("t4x1", "a2c"), ("t4x2", "a2c")]
eval_reports = []
for condition, algorithm in EVAL_CONDITIONS:
    for seed in TRAIN_SEEDS:
        if condition == "cpu":
            checkpoint_root = CPU_ROOT / algorithm / f"seed-{seed}" / algorithm
        else:
            checkpoint_root = GPU_ROOT / condition / f"seed-{seed}" / algorithm
        if not (checkpoint_root / "LATEST").exists():
            print("Skip unavailable checkpoint:", condition, algorithm, seed)
            continue
        output = HELDOUT_ROOT / condition / algorithm / f"train-seed-{seed}.json"
        run([sys.executable, "-m", "validation.evaluate_torch_mvp1",
             "--generation", checkpoint_root, "--algorithm", algorithm,
             "--seed", seed, "--seed-manifest", PREREG_PATH,
             "--split", "validation", "--device", "cpu", "--output", output], cwd=REPO_DIR)
        eval_reports.append(output)
print("Manifest-defined validation reports:", len(eval_reports), "of", len(EVAL_CONDITIONS) * len(TRAIN_SEEDS))
print("Preregistration hash bound:", PREREG_SHA256)


In [ ]:
# Display all policy arms and per-seed cognition contrasts without pooling away seed variation.
import pandas as pd
heldout_rows = []; cognition_rows = []; paired_probe_rows = []
for path in sorted(HELDOUT_ROOT.rglob("*.json")):
    doc = json.loads(path.read_text())
    for row in doc.get("runs", []):
        heldout_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
            "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
            "scenario": row.get("scenario"), "world_seed": row.get("world_seed"), "policy": row.get("policy"),
            "total_reward": row.get("total_reward"), "action_frequencies": row.get("action_frequencies"),
            "mean_policy_entropy_nats": row.get("mean_policy_entropy_nats"),
            "invalid_actions": row.get("invalid_actions"), "navigation_reason_counts": row.get("navigation_reason_counts"),
            "final_drive_means": row.get("final_drive_means")})
    for probe_row in doc.get("paired_cognition_probe_rows", []):
        probe = probe_row.get("probe", {})
        for contrast_name, contrast in probe.get("contrasts", {}).items():
            paired_probe_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
                "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
                "policy": probe_row.get("policy"), "scenario_family": probe_row.get("scenario_family"),
                "world_seed": probe_row.get("world_seed"), "contrast": contrast_name,
                "drink_probability_effect": contrast.get("drink_probability_effect"),
                "ablation_attenuation": contrast.get("ablation", {}).get("attenuation"),
                "input_sha256": contrast.get("input_sha256"), "feature_fidelity": contrast.get("fidelity")} )
    for name, contrast in doc.get("cognition_influence", {}).get("contrasts", {}).items():
        cognition_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
            "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
            "eval_seed": doc.get("seed_split", {}).get("evaluation_root_seed"), "contrast": name,
            "drink_probability_effect": contrast.get("drink_probability_effect"),
            "effect_meets_plus_0_05_margin": contrast.get("effect_meets_plus_0_05_margin"),
            "ablation_attenuation": contrast.get("ablation", {}).get("attenuation"),
            "changed_features": contrast.get("changed_feature_names")})
heldout_df = pd.DataFrame(heldout_rows); cognition_df = pd.DataFrame(cognition_rows)
print("Paired held-out rollouts (raw per world):"); display(heldout_df)
print("Summary first-decision probes (descriptive only):"); display(cognition_df)
print("Paired world-seed cognition probe rows for trained/untrained policies:"); display(pd.DataFrame(paired_probe_rows))


## 10. Resume an interrupted run

The trainer writes atomic checkpoint generations and a `LATEST` pointer. Resume uses the same output directory/config/seed. `--resume latest` restores the latest valid generation; an explicit path may be used to recover a known generation. A resume should continue rather than restart. Do not mix algorithm, device, seed, or config when resuming; trainer compatibility validation should reject mismatches.

In [ ]:
# Example recovery cell. With the same PASTURE_RUN_ID, the full-run cell
# automatically passes --resume latest when a checkpoint exists. This explicit
# cell is useful for recovering one interrupted CPU arm without restarting the
# complete matched loop. Restore ARTIFACT_ROOT from the saved/downloaded run if
# Kaggle did not retain /kaggle/working across a session reset.
RESUME_ALGORITHM = "a2c"
RESUME_DEVICE = "cpu"
RESUME_SEED = TRAIN_SEEDS[0]
RESUME_DIR = CPU_ROOT / RESUME_ALGORITHM / f"seed-{RESUME_SEED}"
RESUME_ENABLED = False  # set True only after confirming run id, source and config
if RESUME_ENABLED:
    train_one(RESUME_ALGORITHM, RESUME_DEVICE, RESUME_SEED, RESUME_DIR, resume="latest")
else:
    print("Resume example is disabled; normal reruns auto-resume existing checkpoints.")


## 11. Inspect training metrics and evaluations

Metrics include rewards/learning curve, per-action distributions and entropy, plus paired perception/memory intervention diagnostics when emitted by the trainer. These are experimental outcomes, not acceptance-gate substitutes. Inspect all seeds, not only favorable runs. Do not modify the frozen MVP1 thresholds or overwrite `validation/MVP1_REMEDIATION_REPORT.md`.

In [ ]:
import pandas as pd
metric_files = list(ARTIFACT_ROOT.rglob("metrics.jsonl"))
rows = []
for path in metric_files:
    try:
        frame = pd.read_json(path, lines=True)
        frame["metrics_path"] = str(path.relative_to(ARTIFACT_ROOT))
        rows.append(frame)
    except ValueError as e:
        print("Could not parse", path, e)
metrics_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print("metrics files:", len(metric_files), "rows:", len(metrics_df))
display(metrics_df.tail(20) if len(metrics_df) else metrics_df)
if len(metrics_df):
    display(metrics_df.describe(include="all").T)

In [ ]:
# Per-seed final evaluations contain action distribution, final physiology,
# and paired memory/perception probability intervention measurements.
evaluation_rows = []
for path in sorted(ARTIFACT_ROOT.rglob("evaluation.json")):
    payload = json.loads(path.read_text())
    evaluation_rows.append({"path": str(path.relative_to(ARTIFACT_ROOT)),
                            "algorithm": payload.get("algorithm"),
                            "seed": payload.get("seed"),
                            "mean_return": payload.get("mean_return"),
                            "episodes": len(payload.get("episodes", [])),
                            "claims_boundary": payload.get("claims_boundary")})
print("Evaluation files:", len(evaluation_rows))
display(pd.DataFrame(evaluation_rows))
for path in sorted(ARTIFACT_ROOT.rglob("evaluation.json"))[:3]:
    payload = json.loads(path.read_text())
    if payload.get("episodes"):
        print(path.relative_to(ARTIFACT_ROOT), "example action frequencies / memory-perception influence")
        display(pd.DataFrame([{"action_frequencies": x.get("action_frequencies"),
                               "memory_delta_l1": x.get("memory_perception_influence", {}).get("memory_delta_l1"),
                               "perception_delta_l1": x.get("memory_perception_influence", {}).get("perception_delta_l1")}
                              for x in payload["episodes"]]))


In [ ]:
# Plot numeric learning curves when the metrics schema exposes common names.
import matplotlib.pyplot as plt
if len(metrics_df):
    xcol = next((c for c in ("episode", "update", "step", "global_step") if c in metrics_df), None)
    ycol = next((c for c in ("episode_reward", "mean_reward", "mean_trajectory_return", "batch_return", "reward", "return") if c in metrics_df), None)
    if xcol and ycol:
        for name, group in metrics_df.groupby("metrics_path"):
            group = group.sort_values(xcol)
            plt.plot(group[xcol], group[ycol], alpha=.65, label=name)
        plt.xlabel(xcol); plt.ylabel(ycol); plt.title("Observed training reward (all emitted runs)")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()
    else:
        print("No recognized reward-column schema; raw metrics remain available for inspection.")

## 12. Export integrity manifest and archive

Kaggle notebook outputs are the persistence boundary. This cell creates SHA-256 entries for every produced artifact (excluding the manifest/archive themselves), writes a manifest, and archives the full run directory for download. Keep this output private unless you intentionally publish synthetic model artifacts. Never push directly to the public source repository.

In [ ]:
import zipfile
export_dir = ARTIFACT_ROOT / "export"
export_dir.mkdir(parents=True, exist_ok=True)
manifest = {"repo_url": REPO_URL, "ref": PASTURE_REF, "commit": commit, "notebook_revision": PASTURE_NOTEBOOK_REVISION, "run_id": RUN_ID, "preregistration_sha256": PREREG_SHA256,
            "python": sys.version, "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(), "files": []}
for path in sorted(ARTIFACT_ROOT.rglob("*")):
    if not path.is_file() or path.name in {"artifact_manifest.json", "pasture-training-artifacts.zip"}:
        continue
    if "export" in path.relative_to(ARTIFACT_ROOT).parts:
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest["files"].append({"path": str(path.relative_to(ARTIFACT_ROOT)),
                              "size_bytes": path.stat().st_size, "sha256": digest})
manifest_path = export_dir / "artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
archive = export_dir / "pasture-training-artifacts.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file() and path not in {archive}:
            zf.write(path, arcname=str(path.relative_to(ARTIFACT_ROOT)))
print("Manifest:", manifest_path, "SHA256:", hashlib.sha256(manifest_path.read_bytes()).hexdigest())
print("Archive:", archive, "bytes:", archive.stat().st_size)
print("Download these files from Kaggle notebook Output files after Save & Run All.")

## 13. Local use and scientific boundaries

Download the exported NPZ + JSON policy pair and its manifest. Verify hashes before use. The inference export is intended for the existing NumPy simulator, not for claiming parity across unrelated versions. Use `farm_sim.policy.load_checkpoint` with the NPZ path after placing the sidecar JSON next to it. Full optimizer/RNG checkpoints are for resuming training and are not the local inference artifact.

**Interpretation:** GPU presence is not acceleration evidence; report measured timings and the plan's independent gates. The environment and reward are synthetic; no cattle biology, production-husbandry, or deployment validity is established.